# Train the LoRA fit judges

Run on a CUDA GPU. This trains the Qwen and DeepSeek LoRA adapters used by `--ranker lora`, using synthetic profiles 1–80. Existing adapters are reused.

On SCU Wave: `sbatch notebooks/wave_notebook.sbatch notebooks/deepseek_lora_evaluation.ipynb`.


In [ ]:
import gc
import random
import sys
from pathlib import Path

import numpy as np
import torch

ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src import ardook_extra_synthetic_data as synthetic
from src.pipeline import OrganizationProfile
from src.fit_judge import (
    DEFAULT_ADAPTER, DEEPSEEK_ADAPTER, DEEPSEEK_JUDGE_MODEL,
    JUDGE_MODEL, LORA_SETTINGS, FitJudgeRanker, QwenEncoder, train_lora_judge,
)

if not torch.cuda.is_available():
    raise RuntimeError('LoRA training requires a CUDA GPU, such as an SCU Wave GPU node.')
print(torch.cuda.get_device_name(0))


In [ ]:
matches = synthetic.load_matches()
relevant = {
    int(profile_id): set(group.loc[group.label == 'match', 'source_row'].astype(int))
    for profile_id, group in matches.groupby('profile_id')
}
profiles = synthetic.load_profiles()
ranker = FitJudgeRanker(
    synthetic.load_grants(), synthetic.load_checker(),
    encoder=QwenEncoder(), judge=lambda prompts: [],
)

positives, negatives = [], []
for row in profiles.itertuples():
    if int(row.id) > 80:
        continue
    profile = OrganizationProfile(row.description, row.attributes)
    for source_row, prompt in ranker.shortlist_prompts(profile, synthetic.REFERENCE_DATE):
        (positives if source_row in relevant[int(row.id)] else negatives).append(prompt)
negatives = random.Random(LORA_SETTINGS['seed']).sample(
    negatives, min(len(negatives), 3 * len(positives))
)
examples = [(prompt, True) for prompt in positives] + [(prompt, False) for prompt in negatives]
print(f'{len(positives)} positive and {len(negatives)} negative examples')


In [ ]:
for model, adapter in ((JUDGE_MODEL, DEFAULT_ADAPTER), (DEEPSEEK_JUDGE_MODEL, DEEPSEEK_ADAPTER)):
    if (adapter / 'adapter_config.json').exists():
        print(f'{adapter}: already trained; keeping it')
        continue
    losses = train_lora_judge(examples, adapter, model_name=model)
    print(f'{adapter}: trained in {len(losses)} steps; final loss {np.mean(losses[-20:]):.4f}')
    gc.collect()
    torch.cuda.empty_cache()
